In [2]:
import geopandas as gpd

In [3]:
import pandas as pd

In [4]:
import os

In [5]:

df = pd.read_csv("dpwh_flood_control_projects.csv")
df = df.dropna(subset=["ProjectLatitude", "ProjectLongitude", "Municipality"])
df["ProjectName"] = df["ProjectName"].str.removeprefix("Construction of ")

gdf_projects = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df.ProjectLongitude, df.ProjectLatitude),
    crs="EPSG:4326",
)



In [6]:
base_dir = "./shapefiles"
gdfs = []

for folder in os.listdir(base_dir):
    folder_path = os.path.join(base_dir, folder)
    if not os.path.isdir(folder_path):
        continue

    shp_files = [f for f in os.listdir(folder_path) if f.lower().endswith(".shp")]
    if not shp_files:
        continue

    shp_path = os.path.join(folder_path, shp_files[0])
    gdf = gpd.read_file(shp_path)
    gdf["SHP_Province"] = folder  # add province column
    gdfs.append(gdf)

    print(f"Loaded {folder} -> {shp_files[0]}")


Loaded Aklan -> PH060400000_FH_5yr.shp
Loaded Albay -> PH050500000_FH_5yr.shp
Loaded Apayao -> Apayao_Flood_5year.shp
Loaded Aurora -> Aurora_Flood_5year.shp
Loaded Bataan -> PH030800000_FH_5yr.shp
Loaded Batangas -> Batangas_Flood_5yr.shp
Loaded Benguet -> Benguet_Flood_5year.shp
Loaded Bohol -> Bohol_Flood_5year.shp
Loaded Bukidnon -> Bukidnon_Flood_5year.shp
Loaded Bulacan -> Bulacan_Flood_5year.shp
Loaded Cagayan -> Cagayan_Flood_5year.shp
Loaded Capiz -> Capiz_Flood_5year.shp
Loaded Catanduanes -> Catanduanes_Flood_5year.shp
Loaded Cavite -> Cavite_Flood_5year.shp
Loaded Cebu -> PH072200000_FH_5yr.shp
Loaded Cotabato -> Cotabato_Flood_5year.shp
Loaded Ifugao -> Ifugao_Flood_5year.shp
Loaded Iloilo -> Iloilo_Flood_5year.shp
Loaded Isabela -> Isabela_Flood_5year.shp
Loaded Kalinga -> Kalinga_Flood_5year.shp
Loaded Laguna -> Laguna_Flood_5year.shp
Loaded Leyte -> PH083700000_FH_5yr.shp
Loaded Maguindanao -> Maguindanao_Flood_5year.shp
Loaded Marinduque -> Marinduque_Flood_5year.shp
L

In [7]:
gdf_flood = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True), crs=gdfs[0].crs)

joined = gpd.sjoin(gdf_projects, gdf_flood, how="left", predicate="intersects")

joined["RiskLevel"] = joined["Var"].fillna(0)
joined["ApprovedBudgetForContract"] = pd.to_numeric(
    joined["ApprovedBudgetForContract"], errors="coerce"
).fillna(0)

joined["ProjectStatus"] = (
    joined["StartDate"].notna() & joined["ActualCompletionDate"].notna()
).map({True: "COMPLETE", False: "INCOMPLETE"})

In [9]:
print(joined.head())
joined.to_file("data/all_joined_projects.gpkg", driver="GPKG")

   MainIsland                            Region Province  \
7       Luzon  Cordillera Administrative Region   Apayao   
8       Luzon  Cordillera Administrative Region   Apayao   
9       Luzon  Cordillera Administrative Region   Apayao   
10      Luzon  Cordillera Administrative Region   Apayao   
11      Luzon  Cordillera Administrative Region   Apayao   

              LegislativeDistrict     Municipality  \
7   APAYAO (LEGISLATIVE DISTRICT)  CONNER (APAYAO)   
8   APAYAO (LEGISLATIVE DISTRICT)  CONNER (APAYAO)   
9   APAYAO (LEGISLATIVE DISTRICT)  CONNER (APAYAO)   
10  APAYAO (LEGISLATIVE DISTRICT)  CONNER (APAYAO)   
11  APAYAO (LEGISLATIVE DISTRICT)  CONNER (APAYAO)   

                 DistrictEngineeringOffice    ProjectId  \
7   Apayao 1st District Engineering Office  P00620988LZ   
8   Apayao 1st District Engineering Office  P00620472LZ   
9   Apayao 1st District Engineering Office  P00620136LZ   
10  Apayao 1st District Engineering Office  P00726398LZ   
11  Apayao 1st Dist

In [10]:
joined.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
Index: 9189 entries, 7 to 9854
Data columns (total 28 columns):
 #   Column                      Non-Null Count  Dtype   
---  ------                      --------------  -----   
 0   MainIsland                  9189 non-null   object  
 1   Region                      9189 non-null   object  
 2   Province                    9189 non-null   object  
 3   LegislativeDistrict         9189 non-null   object  
 4   Municipality                9189 non-null   object  
 5   DistrictEngineeringOffice   9189 non-null   object  
 6   ProjectId                   9189 non-null   object  
 7   ProjectName                 9189 non-null   object  
 8   TypeOfWork                  9189 non-null   object  
 9   FundingYear                 9189 non-null   int64   
 10  ContractId                  9189 non-null   object  
 11  ApprovedBudgetForContract   9189 non-null   float64 
 12  ContractCost                9189 non-null   object  
 13  ActualCompletio

In [18]:
joined["SHP_Province"].value_counts()

SHP_Province
Metro Manila       624
Bulacan            420
Isabela            198
Pampanga           157
Ilocos Norte       139
                  ... 
Capiz                2
Dinagat Islands      2
Quirino              1
Aurora               1
Lanao del Sur        1
Name: count, Length: 63, dtype: int64

In [19]:
joined["Province"].value_counts()

Province
Metro Manila    976
Bulacan         656
Cebu            397
Pangasinan      313
Isabela         313
               ... 
Basilan           8
Quirino           7
Guimaras          7
Sulu              3
Siquijor          1
Name: count, Length: 82, dtype: int64